# Chapter 10: triangle tests, the panel ANOVA and power
The triangle sums are exact binomials. The scoring file holds 144 synthetic scores: 12 assessors x 6 sessions x 2 lagers (18 and 20 IBU), seed 1011.

**No code tier.** `sheets/ch10-triangle.xlsx` does the triangle in one cell: 1 minus BINOM.DIST(correct minus 1, tasters, 1/3, TRUE).

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from scipy import stats

In [ ]:
tail = lambda n, k, p: stats.binom.sf(k - 1, n, p)
print("9 of 18:", round(tail(18, 9, 1 / 3), 4), "| 10 of 18:", round(tail(18, 10, 1 / 3), 4))
lo, hi = stats.beta.ppf(0.025, 9, 10), stats.beta.ppf(0.975, 10, 9)
print(f"95 percent interval on 9 of 18: {lo:.3f} to {hi:.3f}; Pd up to {(3 * hi - 1) / 2:.2f}")
def cut(n): return next(k for k in range(n + 1) if tail(n, k, 1 / 3) <= 0.05)
table = pd.DataFrame([dict(tasters=n, correct_needed=cut(n), false_alarm=round(tail(n, cut(n), 1 / 3), 3),
                           **{f"power Pd {pd_}": round(tail(n, cut(n), 1 / 3 + 2 / 3 * pd_), 2) for pd_ in (0.2, 0.3, 0.4)})
                      for n in (12, 18, 24, 30, 36, 48)])
table

In [ ]:
# duo trio and paired comparison: the same binomial sum with a chance rate of one half (10.2)
one = next(k for k in range(19) if tail(18, k, 0.5) <= 0.05)
two = next(k for k in range(19) if 2 * tail(18, k, 0.5) <= 0.05)
print(f"duo trio or one sided paired, 18 tasters: {one} correct, p = {tail(18, one, 0.5):.3f}")
print(f"two sided paired, 18 tasters: {two} correct, p = {2 * tail(18, two, 0.5):.3f}")
print(f"power of 18 tasters at Pd 0.3: {tail(18, 10, 1 / 3 + 2 / 3 * 0.3):.3f}")

In [ ]:
ns = np.arange(6, 91)
for pd_ in (0.2, 0.3, 0.4): plt.plot(ns, [tail(n, cut(n), 1 / 3 + 2 / 3 * pd_) for n in ns], label=f"Pd {pd_}")
plt.axhline(0.8, ls="--", color="#6b6b6b"); plt.axvline(18, ls=":", color="#c27c0e"); plt.xlabel("tasters"); plt.ylabel("power"); plt.legend(); plt.show()

## The scoring trial: analysis of variance
The sample effect is tested against the assessor x sample line, because the 12 tasters are a sample of the pool of 18.

In [ ]:
s = pd.read_csv(DATA + "ch10_scores.csv"); y = s.bitterness.astype(float); gm = y.mean()
sm, am, tm = (s.groupby(c).bitterness.mean() for c in ("sample", "assessor", "session"))
cell = s.groupby(["assessor", "sample"]).bitterness.mean()
ss_ = {"sample": 72 * ((sm - gm) ** 2).sum(), "assessor": 12 * ((am - gm) ** 2).sum(), "session": 24 * ((tm - gm) ** 2).sum(),
       "assessor x sample": 6 * sum((cell[a, k] - am[a] - sm[k] + gm) ** 2 for a in am.index for k in sm.index)}
ss_["residual"] = ((y - gm) ** 2).sum() - sum(ss_.values())
df_ = {"sample": 1, "assessor": 11, "session": 5, "assessor x sample": 11, "residual": 115}
ms = {k: ss_[k] / df_[k] for k in ss_}
F = {"sample": ms["sample"] / ms["assessor x sample"], "assessor": ms["assessor"] / ms["residual"],
     "session": ms["session"] / ms["residual"], "assessor x sample": ms["assessor x sample"] / ms["residual"]}
dfd = {"sample": 11, "assessor": 115, "session": 115, "assessor x sample": 115}
pd.DataFrame({"SS": ss_, "df": df_, "MS": ms, "F": F, "p": {k: stats.f.sf(F[k], df_[k], dfd[k]) for k in F}}).round(4)

In [ ]:
w = s.pivot_table(index=["assessor", "session"], columns="sample", values="bitterness")
gaps = (w.trial - w.standard).unstack()
per = pd.DataFrame({"mean": am.round(2), "gap": gaps.mean(1).round(2), "t": (gaps.mean(1) / (gaps.std(1) / math.sqrt(6))).round(2)})
per["shows the difference"] = per.t > stats.t.ppf(0.975, 5)
g = gaps.mean(1)
print(f"gap sd {g.std():.3f}; paired t {g.mean() / (g.std() / math.sqrt(12)):.2f}")
per

In [ ]:
def paired_power(n, delta, sd=0.552):
    return stats.nct.sf(stats.t.ppf(0.975, n - 1), n - 1, delta / (sd / math.sqrt(n)))
a, e = (0.914 - 0.650) / 6, 0.650
sessions = 6   # change the number of sessions and tasters here
sd = math.sqrt(2 * (a + e / sessions))
pd.DataFrame({f"gap {d}": [round(paired_power(n, d, sd), 2) for n in (6, 8, 10, 12, 18)] for d in (0.5, 0.75, 1.0)}, index=[6, 8, 10, 12, 18])

**What to look at.** Nine of 18 is not significant and the test had about even odds. Five of 12 tasters show the 2 IBU change on their own; the panel shows it clearly. Change `sessions` to see the floor that more sessions cannot lower.